Import Libraries

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

load datasets

In [4]:
df = pd.read_csv("Data/tickets_dataset.csv")

print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'Data/tickets_dataset.csv'

In [5]:
import os
print(os.getcwd())

d:\Neeru\Python & DataScience\Live projects\AI powered_ service now ticket_intelligence platform\retrieval


In [6]:
import os
print(os.listdir("../"))

['api', 'dashboard', 'data', 'docker', 'models', 'notebooks', 'README.md', 'requirements.txt', 'retrieval', 'src']


In [7]:
print(os.listdir("../data"))

['processed_tickets.csv', 'tickets_dataset.csv']


In [8]:
import pandas as pd

df = pd.read_csv("../data/tickets_dataset.csv")

print(df.head())

   ticket_id                ticket_description         category  priority  \
0       1000    VPN connection failed remotely       VPN Access    Medium   
1       1001  Unable to connect to company VPN       VPN Access  Critical   
2       1002            Form submission failed  Application Bug    Medium   
3       1003  Unable to connect to company VPN       VPN Access    Medium   
4       1004   Password reset link not working   Password Reset       Low   

   resolution_time_hours  assigned_team       status created_date  
0                   2.12     Cloud Team  In Progress   2026-02-04  
1                   3.57     Cloud Team     Resolved   2026-03-05  
2                  10.68  Database Team         Open   2026-06-09  
3                  10.94   Network Team       Closed   2025-09-09  
4                  10.21     Cloud Team         Open   2026-02-17  


In [9]:
print(os.listdir("../data"))

['processed_tickets.csv', 'tickets_dataset.csv']


Step 2 — TF-IDF Vectorization

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create vectorizer
vectorizer = TfidfVectorizer()

# Convert ticket descriptions into vectors
ticket_vectors = vectorizer.fit_transform(df["ticket_description"])

print(ticket_vectors.shape)

(5000, 81)


What this does:

All ticket descriptions → converted into numerical vectors

Example:

"VPN connection failed remotely"
        ↓
[0.12, 0.45, 0.78, ....]

Step 3 — Similarity Function

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_similar_tickets(query):

    # Convert query to vector
    query_vector = vectorizer.transform([query])

    # Compare similarity
    similarity_scores = cosine_similarity(query_vector, ticket_vectors)

    # Top 3 similar tickets
    top_indices = similarity_scores[0].argsort()[-3:][::-1]

    results = df.iloc[top_indices]

    return results

Step 4 — Test Retrieval

In [12]:
query = "VPN connection failed while working remotely"

results = retrieve_similar_tickets(query)

print(results[["ticket_description", "category"]])

                  ticket_description    category
0     VPN connection failed remotely  VPN Access
4142  VPN connection failed remotely  VPN Access
2662  VPN connection failed remotely  VPN Access


TF-IDF (Term Frequency Inverse Document Frequency)

Converts text into numerical vectors

Captures important words in documents

Higher weight for rare but meaningful words

Cosine Similarity

Measures similarity between vectors

Value ranges:

1.0 = exactly similar

0.0 = completely different

Right now you have:

TF-IDF + Cosine Similarity

We will upgrade to:

Sentence Embeddings + FAISS Vector Database

Difference:

TF-IDF	FAISS + Embeddings
Keyword matching	Semantic meaning
Exact word dependency	Understands context
Basic retrieval	Production-grade retrieval
Traditional NLP	Modern GenAI pipeline

In [13]:
!pip install sentence-transformers
!pip install faiss-cpu

Step 2 — Import libraries

In [14]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

c:\Users\hp\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\hp\AppData\Local\Programs\Python\Python39\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


Purpose:

SentenceTransformer → create embeddings

FAISS → fast similarity search

Step 3 — Load dataset

In [15]:
df = pd.read_csv("../data/tickets_dataset.csv")

print(df.head())

   ticket_id                ticket_description         category  priority  \
0       1000    VPN connection failed remotely       VPN Access    Medium   
1       1001  Unable to connect to company VPN       VPN Access  Critical   
2       1002            Form submission failed  Application Bug    Medium   
3       1003  Unable to connect to company VPN       VPN Access    Medium   
4       1004   Password reset link not working   Password Reset       Low   

   resolution_time_hours  assigned_team       status created_date  
0                   2.12     Cloud Team  In Progress   2026-02-04  
1                   3.57     Cloud Team     Resolved   2026-03-05  
2                  10.68  Database Team         Open   2026-06-09  
3                  10.94   Network Team       Closed   2025-09-09  
4                  10.21     Cloud Team         Open   2026-02-17  


Step 4 — Load embedding model

In [17]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded")

W0620 18:05:40.617249 4192 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Embedding model loaded


Step 5 — Convert all tickets to embeddings

In [18]:
ticket_embeddings = embedding_model.encode(
    df["ticket_description"].tolist()
)

print(ticket_embeddings.shape)

(5000, 384)


5000 tickets

each ticket → 384 dimensional vector

Step 6 — Create FAISS index

In [19]:
dimension = ticket_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(ticket_embeddings)
)

print("FAISS index created")

FAISS index created


Meaning:

Store all vectors in searchable vector database

Step 7 — Retrieval function

In [20]:
def retrieve_similar_tickets_faiss(query):

    query_embedding = embedding_model.encode(
        [query]
    )

    distances, indices = index.search(
        np.array(query_embedding),
        k=3
    )

    results = df.iloc[indices[0]]

    return results

What happens:

New query → embedding → FAISS search → top 3 similar tickets

Step 8 — Test

In [21]:
query = "My remote access is not working"

results = retrieve_similar_tickets_faiss(query)

print(
    results[
        ["ticket_description", "category"]
    ]
)

             ticket_description           category
23  Unauthorized access attempt  Security Incident
47  Unauthorized access attempt  Security Incident
54  Unauthorized access attempt  Security Incident


Step 9 — Save FAISS index

Later API will use this.

In [22]:
faiss.write_index(
    index,
    "../models/ticket_faiss.index"
)

print("FAISS index saved")

FAISS index saved


Why FAISS instead of TF-IDF?

Answer:

TF-IDF works based on exact keyword frequency, while FAISS combined with sentence-transformer embeddings captures semantic meaning. Even if wording changes, semantically similar tickets can still be retrieved. This is more suitable for enterprise-scale retrieval systems and modern RAG pipelines.

Embeddings
Convert text into dense vector representation

Example:

"VPN issue"

↓

384 number vector

FAISS
Facebook AI Similarity Search

Used for:

Vector database
Fast nearest neighbor search
RAG retrieval

Semantic Search
Search based on meaning, not exact words

Example:

remote access problem

≈

vpn connection issue

Current Architecture

User raises IT Ticket
          
          ↓
          
Ticket Category Prediction
         
          ↓

Priority Prediction
          
          ↓

Resolution Time Prediction
         
          ↓

Sentence Transformer Embedding
          
          ↓

FAISS Vector Database Search
          
          ↓

Retrieve Similar Historical Tickets
          
          ↓

FastAPI Deployment

Example:

Input:

VPN connection failed while working remotely
Output:

Category: VPN Access

Priority: High

Estimated Resolution Time: 6.3 hours

Similar Historical Tickets Found:
1. VPN login failed
2. Remote VPN disconnected repeatedly

AI Suggested Resolution:

Please verify VPN credentials, restart VPN client,
clear cached network settings and reconnect.
If issue persists contact network administrator.

This becomes RAG (Retrieval Augmented Generation).

"GenAI Response Generation”

Step 1 — Install libraries

In [24]:
!pip install transformers
!pip install torch

Step 2 — Import library

In [25]:
from transformers import pipeline

c:\Users\hp\AppData\Local\Programs\Python\Python39\lib\site-packages\torchvision\datapoints\__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
c:\Users\hp\AppData\Local\Programs\Python\Python39\lib\site-packages\torchvision\transforms\v2\__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Plea

Step 3 — Load LLM model

In [2]:
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

print("LLM Loaded")

c:\Users\hp\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\hp\AppData\Local\Programs\Python\Python39\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
c:\Users\hp\AppData\Local\Programs\Python\Python39\lib\site-packages\torchvision\datapoints\__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some A

LLM Loaded


Step → Generate AI Resolution

Now we test whether the model can generate an answer.

In [10]:
prompt = """
You are an IT support engineer.

Ticket:
VPN connection failed while working remotely.

Provide troubleshooting steps in numbered format.

1.
"""

response = generator(
    prompt,
    max_new_tokens=120
)

print(response[0]["generated_text"])

1.
